In [ ]:
# Cell 1:
# Clone AKOrN model Git repository
# Install requirements and weights.

import os,torch

os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/AKOrN/eval_sudoku.py'):
    !git clone https://github.com/DimitriBnvl/AKOrN.git
    os.chdir('/kaggle/working/AKOrN')
    !pip install -q -r requirements.txt
    !cd data && bash download_satnet.sh && bash download_rrn.sh
os.chdir('/kaggle/working/AKOrN')
print("GPUs:", torch.cuda.device_count(),
      "weights:", os.path.exists('/kaggle/input/datasets/dimitribnvl06/sudoku-akorn-final-weights/ema_99.pth'))

In [ ]:
# Cell 2:
# Checks that the session is ready to start.

import subprocess, os, torch
os.chdir('/kaggle/working/AKOrN')

MP = '/kaggle/input/datasets/dimitribnvl06/sudoku-akorn-final-weights/ema_99.pth'

# 1. GPUs available
ngpu = torch.cuda.device_count()
print(f"GPUs visible: {ngpu}")
assert ngpu >= 2, "Expected 2 GPUs — set accelerator to GPU T4 x2"

# 2. Weights attached
print(f"weights present: {os.path.exists(MP)}")
assert os.path.exists(MP), "Weights .pth not found — is the dataset attached under Input?"

# 3. eval data downloaded
print(f"OOD data present: {os.path.exists('data/sudoku-rrn')}")
assert os.path.exists('data/sudoku-rrn'), "sudoku-rrn missing — Cell 1 download didn't run"

# 4. Code edits present in the committed file
out = subprocess.run(['grep','-cE',
    "SHARD=int|NSHARD=int|MAXB=int|i%NSHARD|totals >= MAXB|corrects_vote=",
    'eval_sudoku.py'], capture_output=True, text=True).stdout.strip()
print(f"eval-flag matches: {out} (want >= 6)")
assert int(out) >= 6, "eval_sudoku.py missing the eval flags — did the fork push go through?"

print("\nALL CHECKS PASSED! Ready to run.")

In [ ]:
# Cell 3:
# Runs the Sudoku model with NBOARDS total boards.

import subprocess, os, re, time
MP='/kaggle/input/datasets/dimitribnvl06/sudoku-akorn-final-weights/ema_99.pth'
NBOARDS  = 128
PER_CARD = NBOARDS // 2

def run_split(K, minimum_chunk, tag):
    args=['python','eval_sudoku.py','--data=ood','--model=akorn',
          f'--model_path={MP}','--T=128',f'--K={K}','--evote_type=sum',
          f'--batchsize={PER_CARD}']
    if minimum_chunk: args.append(f'--minimum_chunk={minimum_chunk}')
    procs, logs = [], []
    for shard in (0,1):
        log=f'/kaggle/working/{tag}_shard{shard}.txt'; logs.append(log)
        env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(shard),
                 SHARD=str(shard), NSHARD='2', MAXB=str(PER_CARD))
        procs.append(subprocess.Popen(args, cwd='/kaggle/working/AKOrN',
                     env=env, stdout=open(log,'w'), stderr=subprocess.STDOUT))
    time.sleep(45)
    os.system('nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv')
    for pr in procs: pr.wait()
    c=t=0
    for log in logs:
        m=re.search(r'corrects_vote=(\d+) totals=(\d+)', open(log).read())
        if m: c+=int(m[1]); t+=int(m[2])
    acc=c/t if t else 0.0
    print(f"[{tag}] K={K}: boards={t} corrects={c} acc={acc:.4f}")
    return K,acc,c,t

# Input to the model.
# K is the number of random iterations per board.
# minimum_chunk is the minimum number of boards per chunk.
results=[run_split(100,None,'k100'), run_split(4096,256,'k4096')]
with open('/kaggle/working/curve_results.txt','w') as f:
    f.write(f"AKOrN Sudoku OOD — same first {NBOARDS} boards, T=128, evote=sum\n")
    for K,acc,c,t in results: f.write(f"K={K}: acc={acc:.4f} ({c}/{t})\n")
print("\n=== CURVE ===\n"+open('/kaggle/working/curve_results.txt').read())